In [3]:
# ========================================
# Часть 1: описание проекта
# ========================================
"""
## Название проекта
NetworkHealthReportAgent

> на основе Hello-Agents + FastAPI +Корпоративная мультистанция для MCPч.сетьЗДОРОВЬЕАнализ& Система вопросов и ответов> Описание: этот блокнот нажимает“Введение в проект -> Настройка среды -> Определение инструмента -> Агентское строительство -> Функциональная демонстрация -> Оценка эффективности ->Краткий прогноз”Упорядочить для удобной презентации и воспроизведения.
## Об авторе
- Имя:monkeyhlj
- GitHub:@monkeyhlj
- дата:2026-05-31
"""

In [4]:
# ========================================
# Часть 2: настройка среды
# ========================================

## Необязательно: при первом запускеУстановка зависимостей(Установлено для пропуска)# !pip install -q hello-agents[all]
# !pip install -q -r requirements.txt

# Импорт библиотек
from datetime import date, timedelta
from pathlib import Path
import os
import json
import time

from dotenv import load_dotenv

# Загрузка переменных среды
load_dotenv()

# Переход в корень проекта
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    # Автопереход в каталог проекта
    PROJECT_ROOT = Path.cwd() / "Co-creation-projects" / "monkeyhlj-NetworkHealthReportAgent"
os.chdir(PROJECT_ROOT)

print("Текущий рабочий каталог:", Path.cwd())

In [6]:
# ========================================
# Часть 3: определение инструментов
# ========================================

from hello_agents.tools.base import Tool, ToolParameter

class SiteQuickLookupTool(Tool):
    """Пример инструмента: быстрый поиск станции по городу."""

    def __init__(self, sites):
        super().__init__(
            name="site_quick_lookup",
            description="Быстрый поиск станций по городу или ключевому слову",
            expandable=False,
        )
        self._sites = sites

    def get_parameters(self):
        return [
            ToolParameter(
                name="query",
                type="string",
                description="Название города, site_id или ключевое слово станции",
                required=True,
            )
        ]

    def run(self, parameters):
        q = (parameters or {}).get("query", "").strip()
        if not q:
            return "Введите название города или ключевое слово станции."
        matched = [
            s for s in self._sites
            if q in s.get("city", "") or q.lower() in s.get("site_id", "").lower() or q in s.get("site_name", "")
        ]
        if not matched:
            return f"Не найдено станций, связанных с {q}."
        return json.dumps(matched, ensure_ascii=False, indent=2)

print("Класс инструмента определён: SiteQuickLookupTool")

In [7]:
# ========================================
# Часть 4: создание агента
# ========================================
# Оркестратор NetworkHealthOrchestrator + опционально SimpleAgent.
# Оркестратор (рекомендуется)
from src.agents.orchestrator import NetworkHealthOrchestrator

orchestrator = NetworkHealthOrchestrator()
sites = orchestrator.list_sites()
runtime = orchestrator.runtime_status()

print("Количество станций:", len(sites))
print("QA Agent:", runtime.get("qa_agent", {}))
print("Первые 3 станции:")
for s in sites[:3]:
    print(f"- {s['site_id']} | {s['site_name']} | {s['city']}")

# SimpleAgent + инструменты
from hello_agents import HelloAgentsLLM, SimpleAgent
from dotenv import load_dotenv
load_dotenv()
api_key = os.getenv("LLM_API_KEY") or os.getenv("OPENAI_API_KEY")
demo_agent = None
if api_key:
    llm_kwargs = {"api_key": api_key}
    base_url = os.getenv("LLM_BASE_URL") or os.getenv("OPENAI_BASE_URL")
    model = os.getenv("LLM_MODEL_ID") or os.getenv("OPENAI_MODEL")
    if base_url:
        llm_kwargs["base_url"] = base_url
    if model:
        llm_kwargs["model"] = model
    llm = HelloAgentsLLM(**llm_kwargs)
    demo_agent = SimpleAgent(
        name="NotebookDemoAgent",
        llm=llm,
        system_prompt="Ты помощник сетевого администрирования; дай краткую рекомендацию на основе вывода инструмента.",
    )
    demo_agent.add_tool(SiteQuickLookupTool(sites))
    print("demo_agent создан и готов к демонстрации.")
else:
    print("LLM_API_KEY/OPENAI_API_KEY не обнаружен — пропуск создания demo_agent.")

In [12]:
# ========================================
# Часть 5: демонстрация функций
# ========================================
"""
Ниже приведено 4 Демо:
1. demo_agent + инструменты (если есть LLM)
2. Отчет о состоянии одного сайта за прошедшую неделю
3. Глобальный Q&A
4. Отчёт по Чэнду и ссылка на скачивание
"""

print("=== Пример 1: demo_agent + Вопросы и ответы по пользовательскому инструменту ===")
# Чтобы избежать пустых результатов, вызванных тем, что модель не запускает вызов инструмента, здесь мы напрямую вызываем инструмент, чтобы получить контекст сайта.
lookup_tool = SiteQuickLookupTool(sites)
cd_context = lookup_tool.run({"query": "成都"})

if demo_agent is None:
    print("demo_agent не создан (обычно из-за отсутствия LLM_API_KEY/OPENAI_API_KEY) — пропуск примера.")
    print("Результат прямого запроса инструмента:\n", cd_context)
else:
    try:
        demo_question = "На основе информации о станции в Чэнду дай одну рекомендацию по эксплуатации.\n\n" + cd_context
        demo_result = demo_agent.run(demo_question)
        print("question: Найди информацию о станции в Чэнду и дай одну рекомендацию по эксплуатации.")
        print("tool_result_preview:\n", cd_context[:300])
        print("answer:\n", str(demo_result)[:600])
    except Exception as e:
        print("Ошибка выполнения demo_agent:", e)
        print("Результат прямого запроса инструмента:\n", cd_context)

print("\n=== Пример 2: Базовый отчет по одному сайту ===")
target_site = sites[0]["site_id"]
end = date.today()
start = end - timedelta(days=6)
report = orchestrator.build_report(site_id=target_site, start=start, end=end)
print("site:", report["site"]["site_name"], report["site"]["site_id"])
print("score:", report["health_score"], "level:", report["health_level"])
print("summary:", report["summary"])

print("\n=== Пример 3: глобальный Q&A ===")
qa_resp = orchestrator.ask_global_question(
    question="Какие site есть в Шанхае?",
    start=start,
    end=end,
    site_id=None,
)
print("answer:\n", qa_resp["answer"][:500])
print("debug:", qa_resp.get("debug", {}))

print("\n=== Пример 4: Экспортировать отчет сайта в Чэнду за последнюю неделю ===")
export_resp = orchestrator.ask_global_question(
    question="Сгенерируй отчёт по site в Чэнду за последнюю неделю",
    start=start,
    end=end,
    site_id=target_site,  #Даже если он идет с другой станции,ч., система также будетпреимущ.Сначала матч по вопросу“Чэнду”
)
print("intent:", export_resp.get("debug", {}).get("intent"))
print("target_site_id:", export_resp.get("debug", {}).get("target_site_id"))
print("artifact:", export_resp.get("artifact"))
print("answer_preview:\n", (export_resp.get("answer") or "")[:800])

In [9]:
# через FastAPI TestClient Демо“Загружаемые ссылки”Поля (аналоговый интерфейсвызов /api/chat）
from fastapi.testclient import TestClient
from src.api.main import app

client = TestClient(app)
payload = {
    "question": "Сгенерируй отчёт по site в Чэнду за последнюю неделю",
    "site_id": "site-bj-hq",
    "start_date": start.strftime("%Y-%m-%d"),
    "end_date": end.strftime("%Y-%m-%d"),
}
resp = client.post("/api/chat", json=payload)
print("status:", resp.status_code)
resp_json = resp.json()
print("intent:", resp_json.get("debug", {}).get("intent"))
print("target_site_id:", resp_json.get("debug", {}).get("target_site_id"))
print("download_url:", resp_json.get("artifact", {}).get("download_url"))

In [10]:
# ========================================
# Часть 6: оценка производительности (опционально)
# ========================================
"""
Замер времени двух путей:
- build_report
- ask_global_question
"""
def benchmark_once(fn, name: str):
    t0 = time.perf_counter()
    result = fn()
    t1 = time.perf_counter()
    print(f"{name} Время: {(t1 - t0):.3f}s")
    return result

_ = benchmark_once(
    lambda: orchestrator.build_report(site_id=target_site, start=start, end=end),
    "build_report",
)

_ = benchmark_once(
    lambda: orchestrator.ask_global_question(
        question="Назови два наиболее рискованных site и объясни причину",
        start=start,
        end=end,
        site_id=None,
    ),
    "ask_global_question",
)

In [11]:
# ========================================
# Часть 7: итоги и перспективы
# ========================================
"""
## Краткое описание проекта

### Функция реализована
- Отчёты по site (логи, устройства, терминалы)
- Карта site + Q&A
- Вопросы и ответы запускают экспорт отчета (вывод в `outputs/`, и можетчерезЗагрузки API)
### Вызовы и решения
-технического обслуживанияч.Распознатьпреимущ.Продвинутый: от“Выбрано”Отрегулируйте“преимущ.Сначала сопоставить по тексту вопроса”
- Ссылка для скачивания не видна: Пройти CORS `expose_headers`Отображение пользовательских заголовков ответов- LLM обязателен для экспорта

### Развитие
- HTML/PDF отчёты
- Тренды (MoM, YoY)
- Объяснимость и траектория инструментов
"""